# Overfitting y Regularización

Una red neuronal puede aprender muy bien los datos de entrenamiento y, aun así, funcionar mal con datos nuevos.

Ese fenómeno se llama **overfitting**.

```text
Training performance      muy buena
Test performance          peor
```

El problema no es que la red no haya aprendido.

El problema es que aprendió demasiado específicamente los ejemplos de entrenamiento.

## Objetivos

Al finalizar podrás:

- reconocer señales de overfitting;
- comparar training y validation/test performance;
- entender qué significa generalización;
- utilizar Dropout;
- entender L2 regularization / weight decay;
- observar el efecto de arquitecturas demasiado grandes;
- explicar por qué regularización puede mejorar la generalización.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn


## 1. Training, Validation y Test

En problemas reales suele ser útil separar:

```text
Training set
    ↓
ajustar parámetros

Validation set
    ↓
tomar decisiones sobre el modelo

Test set
    ↓
evaluación final
```

En este notebook usaremos training y validation para visualizar overfitting.


In [ ]:
torch.manual_seed(42)

wine = load_wine()
X = wine.data
y = wine.target

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.35,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)


## 2. Una red con alta capacidad

Construiremos una red relativamente grande para este dataset pequeño:

```text
13 → 128 → 128 → 64 → 3
```


In [ ]:
model = nn.Sequential(
    nn.Linear(13, 128),
    nn.ReLU(),
    nn.Linear(128, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 3)
)

print("Parameters:", sum(p.numel() for p in model.parameters()))


### Pregunta

¿Por qué una red tan grande podría ser problemática para un dataset pequeño?

<details>
<summary><strong>Pista</strong></summary>

Compara el número de parámetros con el número de observaciones.

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

La red tiene mucha capacidad en relación con el tamaño del dataset.

Eso facilita que pueda ajustar detalles muy específicos del training set en lugar de aprender solamente patrones generales.

</details>


## 3. Seguimos training y validation loss

Entrenaremos durante muchas epochs y guardaremos ambas losses.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 500

train_history = []
val_history = []

for epoch in range(epochs):

    model.train()

    train_logits = model(X_train_t)
    train_loss = criterion(train_logits, y_train_t)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    model.eval()

    with torch.no_grad():
        val_logits = model(X_val_t)
        val_loss = criterion(val_logits, y_val_t)

    train_history.append(train_loss.item())
    val_history.append(val_loss.item())


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_history, label="Training loss")
plt.plot(val_history, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()


## 4. ¿Cómo reconocemos overfitting?

Un patrón típico es:

```text
training loss   ↓ ↓ ↓
validation loss ↓ y luego ↑
```

La red sigue mejorando en training, pero comienza a empeorar en datos no vistos.

<details>
<summary><strong>Mostrar interpretación</strong></summary>

El punto donde validation loss deja de mejorar puede ser una señal de que seguir entrenando ya no ayuda a la generalización.

</details>


## 5. Training y validation accuracy


In [ ]:
def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        pred = torch.argmax(logits, dim=1)
        return (pred == y).float().mean().item()

print("Training accuracy:", accuracy(model, X_train_t, y_train_t))
print("Validation accuracy:", accuracy(model, X_val_t, y_val_t))


Si observamos una diferencia grande entre ambas métricas, debemos investigar posible overfitting.


# 6. Regularización

La **regularización** incluye técnicas que intentan reducir overfitting.

Estudiaremos dos:

1. Dropout
2. L2 regularization / weight decay


## 7. Dropout

Dropout desactiva aleatoriamente una fracción de activaciones durante training.

Por ejemplo:

```python
nn.Dropout(p=0.3)
```

significa que aproximadamente 30% de las activaciones son anuladas durante cada paso de entrenamiento.

Conceptualmente:

```text
Hidden layer

o  o  o  o  o
↓  X  ↓  X  ↓
o     o     o
```

La idea es evitar que la red dependa demasiado de neuronas específicas.


In [ ]:
dropout_model = nn.Sequential(
    nn.Linear(13, 128),
    nn.ReLU(),
    nn.Dropout(p=0.30),

    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(p=0.30),

    nn.Linear(64, 3)
)

print(dropout_model)


## 8. `model.train()` y `model.eval()`

Dropout se comporta diferente durante training y evaluation.

Por eso usamos:

```python
model.train()
```

durante entrenamiento y:

```python
model.eval()
```

durante evaluación.

### Pregunta

¿Por qué no queremos eliminar neuronas aleatoriamente durante la evaluación final?

<details>
<summary><strong>Mostrar solución</strong></summary>

Durante evaluación queremos una salida estable y reproducible usando toda la información aprendida por la red.

Dropout se utiliza como mecanismo de regularización durante training, no para introducir aleatoriedad en la predicción final.

</details>


## 9. Weight Decay

Otra técnica común es penalizar weights muy grandes.

En PyTorch podemos usar:

```python
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)
```

Esto corresponde aproximadamente a una forma de **L2 regularization**.


La idea general es modificar el objetivo:

\[
L_{\mathrm{total}}
=
L_{\mathrm{data}}
+
\lambda
\sum_i w_i^2.
\]

El modelo no solamente intenta reducir el error.

También evita, en cierta medida, weights innecesariamente grandes.


## 10. Entrenamos un modelo regularizado


In [ ]:
torch.manual_seed(42)

regularized_model = nn.Sequential(
    nn.Linear(13, 64),
    nn.ReLU(),
    nn.Dropout(0.30),

    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.30),

    nn.Linear(32, 3)
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    regularized_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

train_reg = []
val_reg = []

for epoch in range(500):

    regularized_model.train()

    logits = regularized_model(X_train_t)
    loss = criterion(logits, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    regularized_model.eval()

    with torch.no_grad():
        val_logits = regularized_model(X_val_t)
        val_loss = criterion(val_logits, y_val_t)

    train_reg.append(loss.item())
    val_reg.append(val_loss.item())


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_reg, label="Training loss")
plt.plot(val_reg, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Regularized Model")
plt.legend()
plt.show()


## 11. Early Stopping

Otra estrategia es detener el entrenamiento cuando validation loss deja de mejorar.

Conceptualmente:

```text
epoch 1    validation improves
epoch 2    validation improves
epoch 3    validation improves
epoch 4    no improvement
epoch 5    no improvement
epoch 6    no improvement
           ↓
        stop
```

A esto lo llamamos **early stopping**.


## 12. Reto

¿Cuál de las siguientes situaciones sugiere overfitting?

### A

```text
Train accuracy = 0.72
Validation accuracy = 0.70
```

### B

```text
Train accuracy = 0.99
Validation accuracy = 0.72
```

<details>
<summary><strong>Mostrar solución</strong></summary>

La situación **B** es mucho más sospechosa de overfitting.

El modelo funciona casi perfectamente en training pero pierde rendimiento considerablemente en datos de validación.

</details>


## 13. Reto de Dropout

Modifica:

```python
nn.Dropout(0.30)
```

por:

```python
nn.Dropout(0.10)
```

y después:

```python
nn.Dropout(0.60)
```

Observa:

- training loss;
- validation loss;
- accuracy.

<details>
<summary><strong>Mostrar interpretación conceptual</strong></summary>

Muy poco Dropout puede no regularizar suficientemente.

Demasiado Dropout puede dificultar el aprendizaje.

El valor apropiado depende del problema y debe evaluarse experimentalmente.

</details>


# Para recordar

Overfitting significa:

```text
memorizar demasiado training data
            ↓
poor generalization
```

Herramientas comunes:

```text
smaller model
dropout
weight decay
more data
data augmentation
early stopping
```

No existe una única estrategia correcta.

Debemos observar training y validation performance juntos.


## Recursos

- [PyTorch — Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)
- [PyTorch — Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)
- [Deep Learning Book — Regularization](https://www.deeplearningbook.org/contents/regularization.html)
